# TRPO et PPO sur HalfCheetah : contraindre les updates de politique

**Algorithmes** : TRPO (Trust Region Policy Optimization) et PPO (Proximal Policy Optimization)
**Environnement** : HalfCheetah-v5 (MuJoCo, Gymnasium)
**Papiers** : [Schulman et al., 2015](https://arxiv.org/abs/1502.05477) (TRPO) ; [Schulman et al., 2017](https://arxiv.org/abs/1707.06347) (PPO)
**Reference pedagogique** : [Spinning Up TRPO](https://spinningup.openai.com/en/latest/algorithms/trpo.html) et [Spinning Up PPO](https://spinningup.openai.com/en/latest/algorithms/ppo.html)

Dans le notebook precedent, A2C-GAE nous a donne une politique actor-critic pour actions continues. Ici, on ajoute une idee centrale : **une politique ne doit pas changer trop brutalement entre deux updates**.

Carte mentale du notebook :

| Concept | Question a laquelle il repond |
|---------|-------------------------------|
| Grand update A2C | Pourquoi un modele qui apprend peut soudainement oublier ? |
| Ancienne vs nouvelle politique | Pourquoi les donnees du rollout ne viennent pas de la politique qu'on optimise apres coup ? |
| Ratio $r_t(\theta)$ | Comment mesurer localement si une action devient plus ou moins probable ? |
| Objectif surrogate | Comment reutiliser les donnees de $\pi_{old}$ pour evaluer $\pi_\theta$ ? |
| KL divergence | Comment mesurer un changement global de comportement ? |
| Trust region | Comment limiter l'update a une zone ou l'approximation reste fiable ? |
| TRPO | Comment imposer explicitement une contrainte KL ? |
| PPO | Comment obtenir presque le meme effet avec un clipping simple et Adam ? |

## HalfCheetah-v5 : l'environnement

HalfCheetah est un robot bipede simplifie dans un plan 2D. L'objectif est de courir le plus vite possible vers la droite.

### Espace d'observation — 17 dimensions

| Groupe | Dimensions | Description |
|--------|-----------|-------------|
| Positions articulaires | 0-5 | Angles des 6 articulations (hanches, genoux, chevilles) |
| Vitesses articulaires | 6-11 | Vitesses angulaires des 6 articulations |
| Vitesse du corps | 12-14 | Vitesse lineaire du centre de masse (x, y, z) |
| Vitesse angulaire | 15-16 | Vitesse angulaire du corps |

### Espace d'actions — 6 dimensions continues

| Dimension | Articulation | Plage |
|-----------|-------------|-------|
| 0 | Hanche avant | $[-1, 1]$ |
| 1 | Genou avant | $[-1, 1]$ |
| 2 | Cheville avant | $[-1, 1]$ |
| 3 | Hanche arriere | $[-1, 1]$ |
| 4 | Genou arriere | $[-1, 1]$ |
| 5 | Cheville arriere | $[-1, 1]$ |

### Structure de la recompense

$$r_t = \underbrace{v_x}_{\text{vitesse vers la droite}} - \underbrace{0.1 \|a_t\|^2}_{\text{penalite d'effort}}$$

La recompense recompense la vitesse horizontale et penalise les efforts energetiques. Il n'y a pas de condition de fin d'episode : l'episode dure toujours 1000 pas.

In [ ]:
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from gymnasium.wrappers import RecordVideo
from torch.distributions import Normal, kl_divergence

try:
    from IPython.display import Video, display
except ImportError:
    Video = None
    display = None

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"] = 100
print(f"PyTorch : {torch.__version__}")
print(f"Gymnasium : {gym.__version__}")

try:
    env_test = gym.make("HalfCheetah-v5")
    env_test.close()
    print("MuJoCo : disponible")
except Exception as e:
    print(f"MuJoCo : ERREUR -> {e}")
    print("Installez avec : pip install gymnasium[mujoco]")


In [ ]:
# === Exploration de l'environnement HalfCheetah ===

env = gym.make("HalfCheetah-v5")
obs, info = env.reset(seed=42)

print("=== Espace d'observation ===")
print(f"  Shape : {env.observation_space.shape}")
print(f"  Bornes basses : {env.observation_space.low[:6]} ...")
print(f"  Bornes hautes : {env.observation_space.high[:6]} ...")

print("\n=== Espace d'actions ===")
print(f"  Shape : {env.action_space.shape}")
print(f"  Bornes : [{env.action_space.low[0]:.1f}, {env.action_space.high[0]:.1f}] pour chaque dim")

print("\n=== Episode aleatoire ===")
obs, _ = env.reset(seed=42)
total_reward = 0.0
for step in range(1000):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

print(f"  Steps : {step + 1}")
print(f"  Recompense totale : {total_reward:.2f}")
print(f"  Derniere observation shape : {obs.shape}")
print(f"  Exemple obs[:5] : {obs[:5]}")
env.close()

## Transition : pourquoi A2C peut mal tourner

Dans A2C, on met a jour la politique directement par descente de gradient :

$$\theta \leftarrow \theta + \alpha \nabla_\theta \mathcal{J}(\theta)$$

Il n'y a aucune contrainte sur l'amplitude du changement. Avec Adam et un learning rate mal regle, ou simplement avec un rollout dont les avantages sont inhabituellement grands, un seul update peut transformer radicalement la politique.

Le probleme : la nouvelle politique est **tres differente** de celle qui a collecte les donnees. Le surrogate de gradient (qui suppose une politique proche) n'est plus une bonne approximation. La politique empiree peut alors produire des rollouts encore plus mauvais, ce qui creuse encore plus l'ecart. L'apprentissage peut diverger.

**Ce qu'on veut** : progresser dans la bonne direction, mais rester proche de la politique actuelle, dans une **zone de confiance** ou notre approximation locale reste fiable.

La suite du notebook construit cette idee etape par etape : ratio de probabilite, surrogate, KL divergence, trust region, et enfin TRPO et PPO.

## Le probleme du grand pas

### La catastrophe A2C

Imaginons que le rollout donne un avantage $\hat{A}_t > 0$ pour une action $a_t$. L'update A2C augmente $\log \pi_\theta(a_t|s_t)$, ce qui rend cette action plus probable. Mais si le learning rate est trop grand ou l'avantage trop eleve, la politique peut changer si drastiquement que :

1. La nouvelle politique produit des actions completement differentes
2. Les donnees du rollout ne sont plus representatives de ce que la nouvelle politique ferait
3. Les estimates de valeur ($V(s)$) deviennent faux pour la nouvelle politique
4. L'episode suivant donne de mauvais rollouts — l'agent a "oublie"

L'update A2C correspond a :

$$\theta_{k+1} = \theta_k + \alpha \cdot \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot \hat{A}_t$$

Il n'existe aucun mecanisme qui garantit que $\pi_{\theta_{k+1}}$ reste proche de $\pi_{\theta_k}$. Un seul mauvais rollout avec des avantages eleves peut detruire des heures d'entrainement.

**La solution** : evaluer $\pi_\theta$ par rapport a l'ancienne politique $\pi_{old}$ via le **ratio de probabilite**.

## Le ratio de probabilite $r_t(\theta)$

### Ancienne vs nouvelle politique

Pendant la collecte, l'agent utilise $\pi_{old}$ (la politique courante). Apres l'update, il optimise $\pi_\theta$ (la nouvelle politique). Ces deux politiques sont differentes, meme si elles partagent la meme architecture.

Le **ratio de probabilite** mesure ce changement action par action :

$$r_t(\theta) = \frac{\pi_\theta(a_t | s_t)}{\pi_{old}(a_t | s_t)}$$

**Interpretation :**
- $r_t = 1$ : l'action $a_t$ a la meme probabilite sous les deux politiques — aucun changement local
- $r_t > 1$ : l'action devient plus probable sous $\pi_\theta$
- $r_t < 1$ : l'action devient moins probable

L'**objectif surrogate** remplace le gradient de log-probabilite par ce ratio :

$$L(\theta) = \mathbb{E}_t \left[ r_t(\theta) \cdot \hat{A}_t \right]$$

Quand $\theta = \theta_{old}$, $r_t = 1$ et $L = \mathbb{E}[\hat{A}_t]$, ce qui recupere exactement l'objectif A2C. L'avantage du surrogate : on peut l'evaluer sans re-collecter de donnees, juste en calculant les nouvelles log-probabilites sur les actions du rollout.

**Mais** : si $r_t$ s'eloigne trop de 1, le surrogate devient une mauvaise approximation de la vraie performance. C'est le probleme qu'on va resoudre.

## L'objectif surrogate et sa limite

Le surrogate non contraint est lineaire en $r_t(\theta)$ :

$$L(\theta) = \mathbb{E}_t \left[ r_t(\theta) \cdot \hat{A}_t \right]$$

Quand l'avantage est positif ($\hat{A}_t > 0$), maximiser $L$ revient a pousser $r_t$ vers $+\infty$ : on veut rendre l'action infiniment plus probable. Quand $\hat{A}_t < 0$, on veut $r_t \to 0$.

Le probleme : **quand $r_t$ s'eloigne de 1, l'approximation devient dangereuse**. Le surrogate a ete construit en supposant que $\pi_\theta \approx \pi_{old}$. Si $r_t = 5$, la nouvelle politique est 5x plus concentree sur cette action — elle est completement differente, et le surrogate n'a plus aucune raison de bien approcher la vraie performance.

La visualisation suivante montre ce comportement : le surrogate croit sans limite, mais la vraie performance stagne ou chute des que la politique change trop.

In [ ]:
# === Intuition visuelle : ratio de probabilite et surrogate non contraint ===

x = np.linspace(-4, 4, 500)
old_dist_1d = Normal(torch.tensor(0.0), torch.tensor(1.0))
new_dist_1d = Normal(torch.tensor(0.55), torch.tensor(1.15))
a_demo = torch.tensor(0.9)

x_t = torch.tensor(x, dtype=torch.float32)
old_pdf = old_dist_1d.log_prob(x_t).exp().numpy()
new_pdf = new_dist_1d.log_prob(x_t).exp().numpy()
ratio_at_action = (new_dist_1d.log_prob(a_demo) - old_dist_1d.log_prob(a_demo)).exp().item()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel gauche : deux distributions, une action
axes[0].plot(x, old_pdf, linewidth=2.5, label=r"$\pi_{old}(a|s)$", color="steelblue")
axes[0].plot(x, new_pdf, linewidth=2.5, linestyle="--", label=r"$\pi_\theta(a|s)$", color="coral")
axes[0].axvline(float(a_demo), color="black", linestyle=":", linewidth=2,
                label=fr"action observee $a_t={float(a_demo):.1f}$")
axes[0].scatter([float(a_demo)], [old_dist_1d.log_prob(a_demo).exp().item()],
                color="steelblue", zorder=3, s=60)
axes[0].scatter([float(a_demo)], [new_dist_1d.log_prob(a_demo).exp().item()],
                color="coral", zorder=3, s=60)
axes[0].set_title(fr"Meme action, deux probabilites : ratio = {ratio_at_action:.2f}")
axes[0].set_xlabel("Action")
axes[0].set_ylabel("Densite")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Panel droit : surrogate non contraint vs ratio
r_vals = np.linspace(0.0, 3.0, 300)
for A, color, label in [(1.0, "coral", r"$\hat{A}_t=+1$"), (-1.0, "steelblue", r"$\hat{A}_t=-1$")]:
    axes[1].plot(r_vals, r_vals * A, color=color, linewidth=2.5, label=label)

axes[1].axvline(1.0, color="black", alpha=0.4, linewidth=1, label="ancienne politique (r=1)")
axes[1].axvspan(0.8, 1.2, color="green", alpha=0.12, label="zone proche")
axes[1].set_title(r"Surrogate non contraint $r_t \hat{A}_t$ : croit sans limite")
axes[1].set_xlabel(r"Ratio $r_t(\theta)$")
axes[1].set_ylabel("Contribution a l'objectif")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Axes :")
print("  Panel gauche : deux gaussiennes, le ratio mesure le rapport des densites a l'action observee")
print("  Panel droit  : le surrogate est lineaire en r — aucune limite pour r >> 1")

## Interpretation : pourquoi contraindre le ratio

Quand $r_t(\theta) \gg 1$ (avec $\hat{A}_t > 0$), le surrogate augmente indefiniment. Mais la vraie performance ne suit pas : la nouvelle politique est si differente de $\pi_{old}$ que les donnees du rollout ne la representent plus.

C'est un probleme d'**importance sampling** : on utilise des donnees collectees sous $\pi_{old}$ pour estimer la performance de $\pi_\theta$. L'estimateur est non-biaise seulement si les deux distributions sont proches. Loin du point de collecte, la variance de l'estimateur explose et le signal devient inutilisable.

**Deux solutions au meme probleme :**
- **TRPO** : contraindre explicitement $D_{KL}(\pi_{old} \| \pi_\theta) \leq \delta$ — resoudre un probleme d'optimisation contraint
- **PPO** : clipper $r_t$ dans $[1-\varepsilon, 1+\varepsilon]$ — eliminer l'incitation a aller loin de 1

Les deux prochaines parties construisent les briques necessaires : acteur gaussien, critique, GAE, et KL divergence.

## Brique 1 : l'acteur gaussien

Pour HalfCheetah (6 dimensions d'action continues), la politique est une **gaussienne diagonale** :

$$\pi_\theta(a|s) = \mathcal{N}(\mu_\theta(s),\ \text{diag}(\sigma^2))$$

- $\mu_\theta(s) \in \mathbb{R}^6$ : la moyenne est la sortie d'un MLP parametrise par $\theta$
- $\sigma \in \mathbb{R}^6$ : l'ecart-type est un parametre appris independamment (pas conditionnel a $s$)

**Pourquoi diagonale ?** On suppose que les 6 dimensions d'action sont independantes conditionnellement a $s$. C'est une approximation commode qui reduit considerablement le nombre de parametres a apprendre.

**Echantillonnage** : $a = \mu_\theta(s) + \sigma \cdot \epsilon$, avec $\epsilon \sim \mathcal{N}(0, I)$ (reparametrage).

**Log-probabilite** : $\log \pi_\theta(a|s) = \sum_{d=1}^{6} \log \mathcal{N}(a_d; \mu_d, \sigma_d)$, somme sur les 6 dimensions.

L'**initialisation orthogonale** de la couche de sortie (gain=0.01) donne des actions initiales proches de zero, ce qui evite des actions extremes au debut de l'entrainement.

In [ ]:
# === GaussianActor : politique gaussienne pour actions continues ===

def orthogonal_init(module, gain=1.0):
    if isinstance(module, nn.Linear):
        nn.init.orthogonal_(module.weight, gain=gain)
        nn.init.zeros_(module.bias)


class GaussianActor(nn.Module):
    """Politique gaussienne pi(a|s; theta) pour actions continues."""

    def __init__(self, obs_dim: int, action_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.mu_head = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        self.apply(lambda m: orthogonal_init(m, gain=np.sqrt(2)))
        orthogonal_init(self.mu_head, gain=0.01)

    def forward(self, x: torch.Tensor):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        mu = self.mu_head(x)
        std = self.log_std.exp().expand_as(mu)
        return Normal(mu, std)

    def get_action(self, obs: np.ndarray):
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        dist = self.forward(obs_t)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(dim=-1)
        return action.squeeze(0).numpy(), log_prob.squeeze(0)

    def evaluate(self, obs_t: torch.Tensor, actions_t: torch.Tensor):
        dist = self.forward(obs_t)
        log_prob = dist.log_prob(actions_t).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        return log_prob, entropy


# --- Mini-test ---
torch.manual_seed(42)
actor_test = GaussianActor(obs_dim=17, action_dim=6)
obs_batch = torch.randn(4, 17)
act_batch = torch.randn(4, 6)

dist = actor_test(obs_batch)
print(f"Forward : mu shape = {dist.mean.shape}, std shape = {dist.stddev.shape}")
print(f"  std initial (log_std=0) : {dist.stddev[0].detach().numpy()}")

action_np, lp = actor_test.get_action(obs_batch[0].numpy())
print(f"\nget_action : action shape = {action_np.shape}, log_prob = {lp.item():.4f}")

log_probs, entropy = actor_test.evaluate(obs_batch, act_batch)
print(f"\nevaluate : log_probs shape = {log_probs.shape}, entropie moy = {entropy.mean().item():.4f}")
print(f"Parametres : {sum(p.numel() for p in actor_test.parameters()):,}")

## Brique 2 : le critique $V(s)$ et le rollout buffer

### Reseau de valeur $V(s; \phi)$

Le critique estime la valeur d'un etat :

$$V_\phi(s) \in \mathbb{R}$$

Meme architecture que l'acteur (MLP 2 couches, tanh), mais sortie scalaire. Il sera entraine par regression sur les **returns bootstrappes** $\hat{R}_t$.

### RolloutBuffer

Contrairement au replay buffer de DQN (off-policy, millions de transitions), le rollout buffer est **on-policy** : il stocke exactement $N$ transitions collectees par la politique courante, utilisees pour un seul update, puis vides.

Structure stockee par transition : $(s_t, a_t, r_t, d_t, V(s_t), \log \pi_{old}(a_t|s_t))$. Le terme $\log \pi_{old}$ est crucial : il permet de calculer le ratio $r_t(\theta)$ plus tard sans rejouer le rollout.

In [ ]:
# === CriticNetwork et RolloutBuffer ===

class CriticNetwork(nn.Module):
    """Reseau de valeur V(s; phi) : observation -> scalaire."""

    def __init__(self, obs_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.value_head = nn.Linear(hidden_dim, 1)
        self.apply(lambda m: orthogonal_init(m, gain=np.sqrt(2)))
        orthogonal_init(self.value_head, gain=1.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        return self.value_head(x).squeeze(-1)


class RolloutBuffer:
    """Buffer on-policy pre-alloue pour n_steps transitions."""

    def __init__(self, n_steps: int, obs_dim: int, action_dim: int):
        self.n_steps = n_steps
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.reset()

    def reset(self):
        self.observations = np.zeros((self.n_steps, self.obs_dim), dtype=np.float32)
        self.actions = np.zeros((self.n_steps, self.action_dim), dtype=np.float32)
        self.rewards = np.zeros(self.n_steps, dtype=np.float32)
        self.dones = np.zeros(self.n_steps, dtype=np.float32)
        self.values = np.zeros(self.n_steps, dtype=np.float32)
        self.log_probs = np.zeros(self.n_steps, dtype=np.float32)
        self.ptr = 0

    def add(self, obs, action, reward, done, value, log_prob):
        self.observations[self.ptr] = obs
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.dones[self.ptr] = done
        self.values[self.ptr] = value
        self.log_probs[self.ptr] = log_prob
        self.ptr += 1

    @property
    def full(self):
        return self.ptr == self.n_steps

    def get_tensors(self):
        return (
            torch.tensor(self.observations, dtype=torch.float32),
            torch.tensor(self.actions, dtype=torch.float32),
            torch.tensor(self.rewards, dtype=torch.float32),
            torch.tensor(self.dones, dtype=torch.float32),
            torch.tensor(self.values, dtype=torch.float32),
            torch.tensor(self.log_probs, dtype=torch.float32),
        )


# --- Mini-tests ---
torch.manual_seed(42)
critic_test = CriticNetwork(obs_dim=17)
obs_batch = torch.randn(8, 17)
values = critic_test(obs_batch)
print(f"CriticNetwork : V(s) shape = {values.shape}, dtype = {values.dtype}")
print(f"  Valeurs exemple : {values.detach().numpy()[:4]}")
print(f"  Parametres : {sum(p.numel() for p in critic_test.parameters()):,}")

buf = RolloutBuffer(n_steps=2048, obs_dim=17, action_dim=6)
print(f"\nRolloutBuffer : {buf.n_steps} steps, obs_dim={buf.obs_dim}, action_dim={buf.action_dim}")
print(f"  Memoire allouee : ~{buf.observations.nbytes // 1024} KB (observations seules)")

## Brique 3 : GAE — avantages generalises

L'avantage $\hat{A}_t$ mesure si une action est meilleure que la moyenne. GAE (Generalized Advantage Estimation) equilibre biais et variance avec le parametre $\lambda$ :

$$\hat{A}_t^{\text{GAE}} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

ou $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ est l'erreur TD a un pas.

**Interpretation :**
- $\lambda = 0$ : $\hat{A}_t = \delta_t$ — avantage TD(0), biais eleve, variance faible
- $\lambda = 1$ : $\hat{A}_t = \sum r_t - V(s_0)$ — retour Monte Carlo, pas de biais, variance elevee
- $\lambda = 0.95$ (valeur standard) : compromis avec tres peu de biais et variance moderee

En pratique, la somme infinie est tronquee au bout du rollout de $N$ steps, avec un bootstrap sur $V(s_N)$ pour estimer les transitions au-dela.

In [ ]:
# === Calcul de GAE : compute_gae ===

def compute_gae(rewards, dones, values, last_value, gamma=0.99, gae_lambda=0.95):
    """Calcule les avantages GAE et les returns bootstrappes.

    rewards, dones, values : tableaux numpy de longueur n_steps
    last_value : V(s_N), bootstrap pour l'etat suivant le rollout
    """
    n_steps = len(rewards)
    advantages = np.zeros(n_steps, dtype=np.float32)
    last_gae = 0.0
    values_extended = np.append(values, last_value)

    for t in reversed(range(n_steps)):
        next_non_terminal = 1.0 - dones[t]
        delta = (rewards[t]
                 + gamma * values_extended[t + 1] * next_non_terminal
                 - values_extended[t])
        last_gae = delta + gamma * gae_lambda * next_non_terminal * last_gae
        advantages[t] = last_gae

    returns = advantages + values
    return advantages, returns


# --- Test numerique ---
np.random.seed(42)
n_test = 10
rewards_test = np.ones(n_test, dtype=np.float32)
dones_test = np.zeros(n_test, dtype=np.float32)
values_test = np.zeros(n_test, dtype=np.float32)

adv_lam0, _ = compute_gae(rewards_test, dones_test, values_test, 0.0, gamma=0.99, gae_lambda=0.0)
adv_lam1, _ = compute_gae(rewards_test, dones_test, values_test, 0.0, gamma=0.99, gae_lambda=1.0)
adv_lam95, _ = compute_gae(rewards_test, dones_test, values_test, 0.0, gamma=0.99, gae_lambda=0.95)

print("GAE avec V(s)=0 partout, recompenses=+1 :")
print(f"  lambda=0.00 (TD-0)   A[0]={adv_lam0[0]:.4f}  A[-1]={adv_lam0[-1]:.4f}")
print(f"  lambda=0.95 (mix)    A[0]={adv_lam95[0]:.4f}  A[-1]={adv_lam95[-1]:.4f}")
print(f"  lambda=1.00 (MC)     A[0]={adv_lam1[0]:.4f}  A[-1]={adv_lam1[-1]:.4f}")
print()
print("  -> lambda=0 : avantage = delta_t = 1 partout (pas de biais)")
print("  -> lambda=1 : avantage = sum(gamma^t * r_t) decroissant vers la fin du rollout")
print("  -> lambda=0.95 : compromis entre les deux")

## Brique 4 : l'agent de base A2CGAEAgent

L'agent de base reunit les trois briques precedentes dans une boucle `learn_step` :

1. **Collecte** : parcourt $N$ steps dans l'environnement avec $\pi_{old}$, stocke les transitions dans le `RolloutBuffer`
2. **GAE** : calcule les avantages et les returns a partir du rollout collecte
3. **`_update`** : methode abstraite, surchargee par `TRPOAgent` et `PPOAgent`

La separation entre collecte et `_update` est la cle de l'architecture : les deux algorithmes partagent exactement la meme collecte de rollout et le meme calcul d'avantages. Seule la procedure d'update de l'acteur change.

L'agent normalise les avantages avant l'update : $\hat{A}_t \leftarrow (\hat{A}_t - \mu) / (\sigma + \epsilon)$. Cette normalisation reduit la sensibilite aux echelles de recompense.

In [ ]:
# === A2CGAEAgent : agent de base commun a TRPO et PPO ===

class A2CGAEAgent:
    """Agent actor-critic avec GAE. Base commune pour TRPO et PPO."""

    def __init__(self, obs_dim, action_dim, hidden_dim=256, lr_critic=3e-4,
                 gamma=0.99, n_steps=2048, gae_lambda=0.95,
                 value_coef=0.5, max_grad_norm=0.5):
        self.gamma = gamma
        self.n_steps = n_steps
        self.gae_lambda = gae_lambda
        self.value_coef = value_coef
        self.max_grad_norm = max_grad_norm

        self.actor = GaussianActor(obs_dim, action_dim, hidden_dim)
        self.critic = CriticNetwork(obs_dim, hidden_dim)
        self.buffer = RolloutBuffer(n_steps, obs_dim, action_dim)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr_critic, eps=1e-5)
        self.history = {"rewards": [], "policy_loss": [], "value_loss": [], "kl": []}
        self._current_obs = None

    def _compute_kl_from_old_dist(self, obs_t, mu_old, std_old):
        """KL(pi_old || pi_new) apres une update de politique."""
        dist_new = self.actor(obs_t)
        dist_old = Normal(mu_old, std_old)
        kl = kl_divergence(dist_old, dist_new).sum(dim=-1).mean()
        return kl.item()

    def _update(self, last_obs, obs_t, actions_t, advantages_t, returns_t, old_log_probs_t):
        raise NotImplementedError

    def learn_step(self, env):
        """Collecte n_steps transitions et effectue une mise a jour."""
        self.buffer.reset()
        self.actor.eval()
        self.critic.eval()

        if self._current_obs is None:
            obs, _ = env.reset()
        else:
            obs = self._current_obs

        episode_rewards = []
        current_ep_reward = 0.0

        for _ in range(self.n_steps):
            with torch.no_grad():
                policy_action, log_prob = self.actor.get_action(obs)
                obs_t_step = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                value = self.critic(obs_t_step).item()

            # policy_action : action brute echantillonnee par la gaussienne.
            # env_action    : action clippee envoyee a HalfCheetah.
            # Les ratios PPO/TRPO doivent rester coherents avec policy_action.
            env_action = np.clip(policy_action, env.action_space.low, env.action_space.high)

            next_obs, reward, terminated, truncated, _ = env.step(env_action)
            done = terminated or truncated
            env_reward = float(reward)
            stored_reward = env_reward

            # TimeLimit bootstrap : une truncation n'est pas un vrai terminal.
            if truncated and not terminated:
                with torch.no_grad():
                    next_obs_t = torch.tensor(next_obs, dtype=torch.float32).unsqueeze(0)
                    terminal_value = self.critic(next_obs_t).item()
                stored_reward = env_reward + self.gamma * terminal_value

            self.buffer.add(
                obs,
                policy_action,       # action brute, pas action clippee
                stored_reward,
                float(done),
                value,
                log_prob.item(),
            )

            current_ep_reward += env_reward
            obs = next_obs

            if done:
                episode_rewards.append(current_ep_reward)
                current_ep_reward = 0.0
                obs, _ = env.reset()

        self._current_obs = obs

        # Pour le logging seulement : garder aussi le morceau d'episode incomplet.
        if current_ep_reward != 0.0:
            episode_rewards.append(current_ep_reward)

        with torch.no_grad():
            last_obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
            last_value = self.critic(last_obs_t).item()

        obs_t, actions_t, rewards_np, dones_np, values_np, log_probs_np = self.buffer.get_tensors()

        advantages_np, returns_np = compute_gae(
            rewards_np.numpy(),
            dones_np.numpy(),
            values_np.numpy(),
            last_value,
            self.gamma,
            self.gae_lambda,
        )

        advantages_t = torch.tensor(advantages_np, dtype=torch.float32)
        returns_t = torch.tensor(returns_np, dtype=torch.float32)
        advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)

        # Les old_log_probs viennent du rollout avant update.
        old_log_probs_t = log_probs_np.detach()

        self.actor.train()
        self.critic.train()

        metrics = self._update(
            obs,
            obs_t,
            actions_t,
            advantages_t,
            returns_t,
            old_log_probs_t,
        )

        metrics["mean_reward"] = np.mean(episode_rewards) if episode_rewards else 0.0

        for key in ["rewards", "policy_loss", "value_loss", "kl"]:
            hist_key = "mean_reward" if key == "rewards" else key
            self.history[key].append(metrics.get(hist_key, 0.0))

        return metrics
# --- Mini-test ---
torch.manual_seed(42)
base_agent = A2CGAEAgent(obs_dim=17, action_dim=6, n_steps=64)
print(f"A2CGAEAgent cree :")
print(f"  Actor  : {sum(p.numel() for p in base_agent.actor.parameters()):,} parametres")
print(f"  Critic : {sum(p.numel() for p in base_agent.critic.parameters()):,} parametres")
print(f"  Buffer : {base_agent.buffer.n_steps} steps")
print(f"  _update() : NotImplementedError -> surchargee par TRPOAgent / PPOAgent")

## KL divergence : mesurer le changement global de comportement

Le ratio $r_t$ regarde une action precise dans un etat precis. Pour une politique gaussienne sur 6 dimensions, on veut aussi mesurer : **la distribution complete est-elle encore proche de l'ancienne ?**

La **divergence KL** mesure cela en integrant sur toutes les actions possibles :

$$D_{KL}(\pi_{old} \| \pi_\theta) = \mathbb{E}_{s \sim \rho^{\pi_{old}}}\left[D_{KL}\left(\pi_{old}(\cdot|s) \| \pi_\theta(\cdot|s)\right)\right]$$

Pour deux gaussiennes diagonales, la KL a une forme analytique :

$$D_{KL}\left(\mathcal{N}(\mu_1, \sigma_1^2) \| \mathcal{N}(\mu_2, \sigma_2^2)\right) = \sum_{d=1}^{D} \left[\log\frac{\sigma_{2,d}}{\sigma_{1,d}} + \frac{\sigma_{1,d}^2 + (\mu_{1,d} - \mu_{2,d})^2}{2\sigma_{2,d}^2} - \frac{1}{2}\right]$$

**Interpretation :**
- $D_{KL} = 0$ : les politiques sont identiques
- $D_{KL}$ petit ($\leq 0.01$) : l'update reste proche, le surrogate est credible
- $D_{KL}$ grand : la politique a trop change, le signal du rollout est obsolete

In [ ]:
# === Visualisation : KL entre politiques gaussiennes ===
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
x = np.linspace(-4, 4, 400)
configs = [
    (0.0, 1.0, 0.0, 1.0, "Identiques"),
    (0.0, 1.0, 0.5, 1.0, "Moyenne decalee (+0.5)"),
    (0.0, 1.0, 0.0, 1.5, "Std change (1.0 -> 1.5)"),
]

for ax, (mu1, s1, mu2, s2, title) in zip(axes, configs):
    p_old = Normal(torch.tensor(mu1), torch.tensor(s1))
    p_new = Normal(torch.tensor(mu2), torch.tensor(s2))
    kl = kl_divergence(p_old, p_new).item()
    x_t = torch.tensor(x, dtype=torch.float32)
    pdf_old = p_old.log_prob(x_t).exp().numpy()
    pdf_new = p_new.log_prob(x_t).exp().numpy()

    ax.plot(x, pdf_old, color="steelblue", linewidth=2.5, label=r"$\pi_{old}$")
    ax.plot(x, pdf_new, color="coral", linewidth=2.5, linestyle="--", label=r"$\pi_\theta$")
    ax.fill_between(x, pdf_old, pdf_new, alpha=0.12, color="purple")
    ax.set_title(f"{title}\nKL = {kl:.4f}")
    ax.set_xlabel("Action")
    ax.set_ylabel("Densite")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("KL entre deux politiques gaussiennes 1D", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Lecture :")
print("  KL=0.0000 : distributions identiques — aucun changement")
print("  KL >0 : la surface coloriee represente la difference de probabilite")
print("  Un decalage de +0.5 sigma donne KL ~ 0.12, bien au-dessus du seuil delta=0.01")

## La trust region : contraindre la KL

La **trust region** est la zone de l'espace des parametres dans laquelle la KL reste sous un seuil $\delta$ :

$$\bar{D}_{KL}(\pi_{old} \| \pi_\theta) \leq \delta$$

avec souvent $\delta = 0.01$.

**Intuition** : a l'interieur de la trust region, l'approximation locale du surrogate reste fiable. Hors de la trust region, l'estimation devient non fiable — on ne peut plus faire confiance a l'objectif surrogate pour predire la vraie performance.

TRPO transforme cette contrainte en un probleme d'optimisation explicite : on cherche le meilleur pas qui ameliore le surrogate tout en restant dans la trust region.

In [ ]:
# === Trust region 2D : heatmap de la contrainte KL ===

delta_mu = np.linspace(-0.7, 0.7, 220)
delta_log_std = np.linspace(-0.45, 0.45, 220)
DM, DLS = np.meshgrid(delta_mu, delta_log_std)

SIGMA_NEW = np.exp(DLS)
KL = np.log(SIGMA_NEW) + (1.0 + DM**2) / (2.0 * SIGMA_NEW**2) - 0.5
SURR = 1.0 * DM + 0.35 * DLS - 0.08 * (DM**2 + DLS**2)

a2c_step = np.array([0.50, 0.22])
trpo_step = np.array([0.12, 0.035])

fig, ax = plt.subplots(figsize=(8, 6))
cont = ax.contourf(DM, DLS, SURR, levels=24, cmap="viridis", alpha=0.85)
plt.colorbar(cont, ax=ax, label="Surrogate (plus clair = mieux)")
trust = ax.contour(DM, DLS, KL, levels=[0.01], colors="white", linewidths=3)
ax.clabel(trust, fmt={0.01: "KL = 0.01"}, colors="white", fontsize=9)
ax.contour(DM, DLS, KL, levels=[0.05, 0.10], colors="white", linestyles="--", alpha=0.5)

ax.scatter([0], [0], color="black", s=70, zorder=5, label="politique actuelle")
ax.annotate("", xy=(a2c_step[0], a2c_step[1]), xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color="red", lw=2.5))
ax.annotate("A2C\n(hors trust region)", xy=a2c_step,
            xytext=(a2c_step[0]+0.05, a2c_step[1]+0.06), color="red", fontsize=9)
ax.annotate("", xy=(trpo_step[0], trpo_step[1]), xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color="lime", lw=2.5))
ax.annotate("TRPO\n(dans la trust region)", xy=trpo_step,
            xytext=(trpo_step[0]+0.05, trpo_step[1]-0.08), color="lime", fontsize=9)

ax.set_xlabel("Delta moyenne (delta_mu)")
ax.set_ylabel("Delta log-std (delta_log_std)")
ax.set_title("Trust region KL <= 0.01 dans l'espace des parametres")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Interpretation : trust region dans l'espace des parametres

La heatmap montre :

- **Couleur** : valeur du surrogate local — plus clair = meilleur objectif
- **Contour blanc continu** : frontiere de la trust region ($D_{KL} = 0.01$)
- **Fleche rouge (A2C)** : le gradient non contraint va loin dans la direction du gradient, hors de la trust region
- **Fleche verte (TRPO)** : le pas TRPO reste a l'interieur de la trust region, dans la zone ou l'approximation est fiable

Le gradient non contraint (A2C) monte bien l'objectif surrogate local — mais la nouvelle politique est si differente de l'ancienne que le surrogate n'est plus une bonne approximation de la vraie performance.

TRPO fait un pas plus petit, mais **dans une zone ou on peut avoir confiance** que le surrogate predit correctement ce que fera la vraie politique.

## Transition : deux approches pour contraindre le pas

On a maintenant tous les elements pour comprendre TRPO et PPO :

- **Le surrogate** $L(\theta) = \mathbb{E}_t[r_t(\theta)\hat{A}_t]$ est l'objectif a maximiser
- **La trust region** $D_{KL} \leq \delta$ est la contrainte a respecter
- **Les deux algorithmes resolvent ce probleme de deux facons radicalement differentes**

**TRPO** : resoudre le probleme contraint exactement. Calculer la direction optimale via gradient conjugue, puis valider le pas par une line search qui verifie la contrainte KL numeriquement.

**PPO** : modifier l'objectif pour que l'optimiseur (Adam) ne puisse pas naturellement aller loin. Clipper $r_t(\theta)$ dans $[1-\varepsilon, 1+\varepsilon]$ supprime l'incitation a s'eloigner de la politique ancienne.

Les deux prochaines parties implementent ces deux approches.

## TRPO : le probleme d'optimisation contraint

TRPO formule l'update de la politique comme un probleme d'optimisation contraint :

$$\max_\theta \quad L(\theta) = \mathbb{E}_t\left[r_t(\theta)\hat{A}_t\right] \quad \text{s.t.} \quad \bar{D}_{KL}(\pi_{old} \| \pi_\theta) \leq \delta$$

C'est la traduction mathematique directe de : **ameliore la politique, mais ne sors pas de la trust region**.

**Pourquoi on ne resout pas directement ce probleme ?** La KL est non-lineaire en $\theta$. Former le Lagrangien et le resoudre exactement demande de former la matrice hessienne de la KL — une matrice de taille $|\theta| \times |\theta|$ qui peut avoir des centaines de millions d'entrees. Completement infaisable.

## Approximation locale et gradient naturel

Autour de $\theta_{old}$, on approxime le probleme avec des developpements du premier et second ordre. Si $d = \theta - \theta_{old}$ :

$$L(\theta_{old}+d) \approx L(\theta_{old}) + g^T d \quad \text{(approximation lineaire)}$$

ou $g = \nabla_\theta L$ est le gradient du surrogate et $F$ est la **matrice de Fisher** (hessien de la KL) :

$$D_{KL}(\pi_{old} \| \pi_{\theta_{old}+d}) \approx \frac{1}{2}d^T F d$$

La solution de ce probleme approxime a la forme du **gradient naturel** : $d^* = F^{-1} g$.

Le gradient naturel corrige la direction de descente en tenant compte de la geometrie de la distribution. Contrairement au gradient ordinaire $g$, $F^{-1}g$ tient compte du fait que des petits changements de parametres peuvent produire de grands changements de comportement.

**Pourquoi ne pas former $F$ explicitement ?** Si l'actor a $P$ parametres, $F$ est une matrice $P \times P$. Pour $P = 200\,000$, cela represente $4 \times 10^{10}$ valeurs — impossible en memoire. TRPO utilise le **gradient conjugue** pour calculer $F^{-1}g$ sans former $F$.

In [ ]:
# === Produit Fisher-vecteur et gradient conjugue ===

def fisher_vector_product(actor, obs_t, vector, damping=0.1):
    """Calcule Fv = nabla(nabla KL . v) par double backprop.

    F : matrice de Fisher (hessien de la KL)
    damping : regularisation pour stabiliser CG (ajoute damping*I)
    """
    with torch.no_grad():
        dist_old = actor(obs_t)
        mu_old = dist_old.mean.detach()
        std_old = dist_old.stddev.detach()

    dist_new = actor(obs_t)
    dist_old_fixed = Normal(mu_old, std_old)
    kl = torch.distributions.kl_divergence(dist_old_fixed, dist_new).sum(dim=-1).mean()

    grad_kl = torch.autograd.grad(kl, actor.parameters(), create_graph=True)
    grad_kl_flat = torch.cat([g.reshape(-1) for g in grad_kl])

    kl_v = (grad_kl_flat * vector).sum()

    Fv = torch.autograd.grad(kl_v, actor.parameters())
    Fv_flat = torch.cat([g.reshape(-1) for g in Fv]).detach()
    return Fv_flat + damping * vector


def conjugate_gradient(fvp_fn, b, n_steps=10, tol=1e-10):
    """Resout Fx = b par gradient conjugue sans former F.

    fvp_fn : fonction v -> Fv (produit Fisher-vecteur)
    b      : membre droit (gradient surrogate)
    """
    x = torch.zeros_like(b)
    r = b.clone()
    p = b.clone()
    r_dot = torch.dot(r, r)
    residuals = [r_dot.item()]

    for _ in range(n_steps):
        Fp = fvp_fn(p)
        alpha = r_dot / (torch.dot(p, Fp) + 1e-8)
        x = x + alpha * p
        r = r - alpha * Fp
        r_dot_new = torch.dot(r, r)
        residuals.append(r_dot_new.item())
        if r_dot_new < tol:
            break
        beta = r_dot_new / (r_dot + 1e-8)
        p = r + beta * p
        r_dot = r_dot_new

    return x, residuals


# --- Demo : resoudre Ax = b pour une matrice SPD ---
torch.manual_seed(42)
n = 50
A_rand = torch.randn(n, n)
A_spd = A_rand @ A_rand.T + torch.eye(n) * 0.1
b_vec = torch.randn(n)

x_exact = torch.linalg.solve(A_spd, b_vec)
x_cg, res = conjugate_gradient(lambda v: A_spd @ v, b_vec, n_steps=50)

residual = (A_spd @ x_cg - b_vec).norm().item()
error = (x_cg - x_exact).norm().item()
print(f"Demo CG (n={n}, 50 iterations) :")
print(f"  Residu  ||Ax - b|| = {residual:.2e}")
print(f"  Erreur  ||x_cg - x_exact|| = {error:.2e}")
print(f"  -> CG resout Ax=b sans former A (economie : {n}^2 = {n**2} entrees evitees)")

In [ ]:
# === Visualisation : convergence du gradient conjugue ===

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(res, color="purple", linewidth=2.5, marker="o", markersize=4)
ax.axhline(1e-6, color="red", linestyle="--", alpha=0.5, label="tol = 1e-6")
ax.set_xlabel("Iteration CG")
ax.set_ylabel("Residu ||r||^2 (echelle log)")
ax.set_title(f"Convergence CG : {len(res)} iterations pour resoudre Fx = b (n={n})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Iterations CG : {len(res)}")
print(f"  Residu initial : {res[0]:.4f}")
print(f"  Residu final   : {res[-1]:.2e}")

## Interpretation : gradient conjugue pour la direction TRPO

La courbe de convergence montre que le gradient conjugue reduit le residu de plusieurs ordres de grandeur en quelques dizaines d'iterations — sans jamais former la matrice $F$.

Pour TRPO avec un actor de 200k parametres :
- Former $F$ explicitement : $200000^2 \times 4$ octets $\approx 160$ GB — impossible
- CG avec 10 iterations : 10 evaluations du produit $Fv$ (double backprop) — quelques secondes

Chaque iteration CG ne demande qu'un produit $Fv$, qui se calcule en $O(|\theta|)$ par double differentiation automatique. C'est l'astuce cle de TRPO : resoudre un probleme quadratique de grande dimension sans jamais stocker la matrice.

## TRPO : line search pour valider le pas

Le gradient conjugue donne une **direction** $d^* \approx F^{-1}g$. Mais la direction ne dit pas combien avancer. L'approximation quadratique de la KL peut etre inexacte loin du point de depart.

TRPO verifie le pas candidat directement en calculant la vraie KL apres l'avoir applique. Le step doit satisfaire deux conditions :
1. **Le surrogate augmente** : $L(\theta_{new}) > L(\theta_{old})$
2. **La KL reste sous le seuil** : $D_{KL}(\pi_{old} \| \pi_{new}) \leq \delta$

### Backtracking line search

Le plus grand pas theorique est $\alpha_0 = \sqrt{2\delta / (d^T F d)}$. On essaie ensuite :

$$\theta_{candidate} = \theta_{old} + \alpha_0 \cdot \beta^k \cdot d^*$$

avec $\beta = 0.5$ et $k = 0, 1, 2, \ldots$

Si les deux conditions sont remplies, on accepte le pas. Sinon, on divise $\alpha$ par 2 et on reessaie. Si aucun pas ne marche apres $K$ tentatives, on annule l'update de l'acteur.

In [ ]:
# === Helpers : flat params et backtracking line search ===

def get_flat_params(model):
    """Recupere tous les parametres du modele en un vecteur plat."""
    return torch.cat([p.data.reshape(-1) for p in model.parameters()])


def set_flat_params(model, flat_params):
    """Recharge les parametres du modele depuis un vecteur plat."""
    offset = 0
    for p in model.parameters():
        numel = p.numel()
        p.data.copy_(flat_params[offset:offset + numel].reshape(p.shape))
        offset += numel


def flat_grad(loss, model, retain_graph=False):
    """Calcule le gradient et le retourne en vecteur plat."""
    grads = torch.autograd.grad(loss, model.parameters(), retain_graph=retain_graph)
    return torch.cat([g.reshape(-1) for g in grads])


def backtracking_line_search(actor, obs_t, actions_t, advantages_t, old_log_probs_t,
                              step_dir, full_step, max_kl=0.01,
                              backtrack_coeff=0.5, max_backtracks=10):
    """Trouve le step qui ameliore L et respecte KL <= max_kl."""
    with torch.no_grad():
        log_probs_old, _ = actor.evaluate(obs_t, actions_t)
        old_surr = (torch.exp(log_probs_old - old_log_probs_t) * advantages_t).mean().item()

    old_params = get_flat_params(actor)

    with torch.no_grad():
        dist_old = actor(obs_t)
        mu_old = dist_old.mean.clone()
        std_old = dist_old.stddev.clone()

    for i in range(max_backtracks):
        alpha = full_step * (backtrack_coeff ** i)
        set_flat_params(actor, old_params + alpha * step_dir)

        with torch.no_grad():
            log_probs_new, _ = actor.evaluate(obs_t, actions_t)
            new_surr = (torch.exp(log_probs_new - old_log_probs_t) * advantages_t).mean().item()
            dist_new = actor(obs_t)
            kl = torch.distributions.kl_divergence(
                Normal(mu_old, std_old), dist_new
            ).sum(dim=-1).mean().item()

        if new_surr > old_surr and kl <= max_kl:
            print(f"    Line search OK : backtrack={i}, alpha={alpha:.4f}, "
                  f"surr: {old_surr:.4f}->{new_surr:.4f}, KL={kl:.5f}")
            return True

    set_flat_params(actor, old_params)
    print("    Line search echouee : reversion aux anciens parametres")
    return False


# --- Demo ---
torch.manual_seed(42)
actor_demo = GaussianActor(17, 6)
params_before = get_flat_params(actor_demo).clone()
set_flat_params(actor_demo, params_before + 0.01)
params_after = get_flat_params(actor_demo)
set_flat_params(actor_demo, params_before)
print(f"get/set flat params :")
print(f"  Taille vecteur plat : {params_before.shape[0]:,} parametres")
print(f"  Perturbation +0.01 : delta_norm = {(params_after - params_before).norm().item():.4f}")
print(f"  Apres restauration : delta = {(get_flat_params(actor_demo) - params_before).norm().item():.2e}")

## Transition : TRPO fonctionne, mais a quel prix ?

TRPO donne une mise a jour rigoureuse avec une garantie theorique de ne pas sortir de la trust region. Mais chaque update de l'acteur demande :

1. Un backward pass complet pour $g$ (gradient du surrogate)
2. 10 iterations CG, chacune avec un produit $Fv$ (double backprop)
3. Jusqu'a 10 evaluations de line search

Au total, chaque update TRPO est 5 a 10 fois plus couteux qu'un simple gradient step Adam.

De plus, TRPO est difficile a adapter a des architectures complexes : reseaux recurrents, parametres partages entre acteur et critique, couches de normalisation. Le CG suppose un espace de parametres lisse.

PPO propose une alternative radicalement plus simple : **modifier l'objectif** pour que Adam ne puisse pas naturellement aller loin. Pas de CG, pas de line search, juste un clip.

## PPO : clipper le ratio plutot que contraindre la KL

L'idee de PPO est simple : si le probleme est que $r_t(\theta)$ peut s'eloigner de 1, il suffit de **supprimer l'incitation a aller loin**. On remplace le surrogate non contraint par un surrogate clippe :

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta)\hat{A}_t,\ \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)\hat{A}_t \right) \right]$$

avec typiquement $\varepsilon = 0.2$. Le clipping force $r_t$ dans $[0.8, 1.2]$ pour les calculs de gradient — au-dela de ces bornes, le gradient est nul (le clip est actif), donc Adam ne peut pas pousser le ratio plus loin.

## Comment le clip protege la politique

Le $\min$ dans la loss PPO joue un role essentiel. Analysons les deux cas :

**Cas $\hat{A}_t > 0$ (action meilleure que la moyenne)** : on veut augmenter $r_t$. Le clip bloque a $1+\varepsilon$. Au-dela, le gradient devient nul — plus d'incitation a aller plus loin.

**Cas $\hat{A}_t < 0$ (action moins bonne)** : on veut diminuer $r_t$. Le clip bloque a $1-\varepsilon$. En-dessous, le gradient s'annule aussi.

L'objectif PPO complet combine cette loss avec le terme de valeur :

$$\mathcal{L}^{PPO}(\theta) = L^{\text{CLIP}}(\theta) - c_v \mathcal{L}^V(\phi) + c_e H[\pi_\theta]$$

ou $c_v$ penalise l'erreur du critique et $c_e$ encourage l'entropie (exploration).

In [ ]:
# === Visualisation : surrogate PPO clippe ===

r_vals = np.linspace(0.0, 2.5, 300)
clip_eps = 0.2
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, A, title_suffix in zip(axes, [1.0, -1.0],
                                ["A > 0 : action avantageuse", "A < 0 : action desavantageuse"]):
    surr_unclipped = r_vals * A
    surr_clipped = np.clip(r_vals, 1 - clip_eps, 1 + clip_eps) * A
    ppo_loss = np.minimum(surr_unclipped, surr_clipped)

    ax.plot(r_vals, surr_unclipped, color="coral", linewidth=2, linestyle="--",
            label="Surrogate non clippe")
    ax.plot(r_vals, surr_clipped, color="steelblue", linewidth=2, linestyle=":",
            label="Surrogate clippe")
    ax.plot(r_vals, ppo_loss, color="purple", linewidth=3, label="PPO : min des deux")
    ax.axvline(1.0, color="black", alpha=0.3, linewidth=1)
    ax.axvline(1 - clip_eps, color="green", alpha=0.5, linestyle="--", linewidth=1,
               label=f"bornes clip [{1-clip_eps:.1f}, {1+clip_eps:.1f}]")
    ax.axvline(1 + clip_eps, color="green", alpha=0.5, linestyle="--", linewidth=1)
    ax.set_xlabel(r"Ratio $r_t(\theta)$")
    ax.set_ylabel("Contribution a l'objectif")
    ax.set_title(f"Loss PPO : {title_suffix}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Lecture :")
print("  A > 0 : le gradient est nul pour r > 1+eps — plus d'incitation a augmenter r")
print("  A < 0 : le gradient est nul pour r < 1-eps — plus d'incitation a diminuer r")
print("  PPO garde le min : toujours conservateur, jamais optimiste au-dela des bornes")

## Interpretation : l'effet du clip en pratique

La loss PPO est **toujours conservatrice** : elle ne recompense jamais un ratio loin de 1. En pratique :

- Si Adam essaie de pousser $r_t > 1+\varepsilon$ (cas $A > 0$), le gradient PPO devient zero — Adam s'arrete naturellement
- Si Adam essaie de pousser $r_t < 1-\varepsilon$ (cas $A < 0$), idem

Le clip ne garantit pas que $r_t$ reste dans $[1-\varepsilon, 1+\varepsilon]$ : il est possible que le ratio depasse les bornes apres K epochs de mini-batches. Mais l'**early stopping** sur la KL (`target_kl=0.01`) arrete l'epoch si la KL devient trop grande, ce qui assure une contrainte pratique supplementaire.

PPO combine ainsi deux mecanismes : le clip (supprime l'incitation) et le early stopping (coupe si la KL deborde quand meme).

## TRPO vs PPO : deux approches du meme probleme

| Dimension | TRPO | PPO |
|-----------|------|-----|
| **Contrainte** | KL exacte ($D_{KL} \leq \delta$, garantie) | Clipping $r_t \in [1-\varepsilon, 1+\varepsilon]$ + early stop |
| **Optimiseur actor** | Gradient conjugue + backtracking line search | Adam standard |
| **Epochs par rollout** | 1 (un seul step CG+LS) | K (typiquement 10) |
| **Cout computationnel** | Eleve (double backprop, CG, line search) | Standard (un backward pass) |
| **Garantie theorique** | Monotonie approximee | Aucune formelle |
| **Implementation** | Complexe (200+ lignes) | Simple (20 lignes de loss) |
| **Efficacite en donnees** | Faible | Bonne (K reutilisations du rollout) |
| **Standard industriel** | Non | Oui (SB3, CleanRL, OpenAI) |

Les deux algorithmes partagent la meme base : meme collecte de rollout, meme calcul GAE, meme architecture acteur-critique. Seule `_update()` change.

In [ ]:
# === TRPOAgent : surcharge de _update() avec CG + line search ===

class TRPOAgent(A2CGAEAgent):
    """Agent TRPO : update de l'actor par gradient conjugue + line search.

    Herite de A2CGAEAgent. Seule difference : la methode _update().
    Le critic est entraine separement avec Adam.
    """

    def __init__(self, *args, max_kl=0.01, cg_steps=10, cg_damping=0.1,
                 backtrack_coeff=0.5, max_backtracks=10, critic_epochs=5, **kwargs):
        super().__init__(*args, **kwargs)
        self.max_kl = max_kl
        self.cg_steps = cg_steps
        self.cg_damping = cg_damping
        self.backtrack_coeff = backtrack_coeff
        self.max_backtracks = max_backtracks
        self.critic_epochs = critic_epochs

    def _fvp(self, obs_t, vector):
        return fisher_vector_product(self.actor, obs_t, vector, damping=self.cg_damping)

    def _update(self, last_obs, obs_t, actions_t, advantages_t, returns_t, old_log_probs_t):
        """Update TRPO : 1) gradient, 2) CG, 3) line search."""
        with torch.no_grad():
            dist_old = self.actor(obs_t)
            mu_old = dist_old.mean.clone()
            std_old = dist_old.stddev.clone()
            
        log_probs, _ = self.actor.evaluate(obs_t, actions_t)
        ratio = torch.exp(log_probs - old_log_probs_t)
        surr_loss = -(ratio * advantages_t).mean()
        g = flat_grad(surr_loss, self.actor, retain_graph=False)

        fvp_fn = lambda v: self._fvp(obs_t, v)
        step_dir, _ = conjugate_gradient(fvp_fn, -g, n_steps=self.cg_steps)

        Fd = self._fvp(obs_t, step_dir)
        dFd = torch.dot(step_dir, Fd)
        if dFd.item() <= 0:
            return {"policy_loss": 0.0, "value_loss": 0.0, "kl": 0.0}
        full_step = torch.sqrt(2 * self.max_kl / (dFd + 1e-8)).item()

        backtracking_line_search(
            self.actor, obs_t, actions_t, advantages_t, old_log_probs_t,
            step_dir=step_dir, full_step=full_step,
            max_kl=self.max_kl, backtrack_coeff=self.backtrack_coeff,
            max_backtracks=self.max_backtracks,
        )

        kl_after = self._compute_kl_from_old_dist(obs_t, mu_old, std_old)

        value_losses = []
        for _ in range(self.critic_epochs):
            values_pred = self.critic(obs_t)
            value_loss = F.mse_loss(values_pred, returns_t)
            self.critic_optimizer.zero_grad()
            value_loss.backward()
            nn.utils.clip_grad_norm_(self.critic.parameters(), self.max_grad_norm)
            self.critic_optimizer.step()
            value_losses.append(value_loss.item())

        with torch.no_grad():
            log_probs_new, _ = self.actor.evaluate(obs_t, actions_t)
            ratio_new = torch.exp(log_probs_new - old_log_probs_t)
            policy_loss_after = -(ratio_new * advantages_t).mean().item()

        return {
            "policy_loss": policy_loss_after,
            "value_loss": np.mean(value_losses),
            "kl": kl_after,
        }

In [ ]:
# === Mini-test TRPOAgent ===

torch.manual_seed(42)
trpo_test = TRPOAgent(obs_dim=17, action_dim=6, n_steps=256, max_kl=0.01)

print(f"TRPOAgent :")
print(f"  Actor  : {sum(p.numel() for p in trpo_test.actor.parameters()):,} parametres")
print(f"  Critic : {sum(p.numel() for p in trpo_test.critic.parameters()):,} parametres")
print(f"  max_kl={trpo_test.max_kl}, cg_steps={trpo_test.cg_steps}")

obs_v = torch.randn(256, 17)
act_v = torch.randn(256, 6) * 0.5
adv_v = torch.randn(256)
adv_v = (adv_v - adv_v.mean()) / (adv_v.std() + 1e-8)
ret_v = adv_v + 1.0

with torch.no_grad():
    old_lp_v, _ = trpo_test.actor.evaluate(obs_v, act_v)

with torch.no_grad():
    dist_before = trpo_test.actor(obs_v)
    mu_before = dist_before.mean.clone()
    std_before = dist_before.stddev.clone()

print("\nExecution _update (line search verbose) :")
metrics = trpo_test._update(np.zeros(17), obs_v, act_v, adv_v, ret_v, old_lp_v)

with torch.no_grad():
    dist_after = trpo_test.actor(obs_v)
    kl_exact = torch.distributions.kl_divergence(
        Normal(mu_before, std_before), dist_after
    ).sum(-1).mean().item()

print(f"\nKL apres update (exacte) : {kl_exact:.5f}")
print(f"Contrainte respectee : {kl_exact <= trpo_test.max_kl}  (KL={kl_exact:.5f} <= delta=0.01)")
print(f"Metrics : {metrics}")

In [ ]:
# === PPOAgent : surcharge de _update() avec clipping + Adam ===

class PPOAgent(A2CGAEAgent):
    """Agent PPO : update de l'actor par loss clippee et Adam.

    Herite de A2CGAEAgent. Seule difference : la methode _update().
    K epochs de mini-batch SGD avec early stopping sur KL.
    """

    def __init__(self, *args, lr_actor=3e-4, clip_eps=0.2,
                 ppo_epochs=10, minibatch_size=64,
                 target_kl=0.01, entropy_coef=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.clip_eps = clip_eps
        self.ppo_epochs = ppo_epochs
        self.minibatch_size = minibatch_size
        self.target_kl = target_kl
        self.entropy_coef = entropy_coef
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor, eps=1e-5)

    def _update(self, last_obs, obs_t, actions_t, advantages_t, returns_t, old_log_probs_t):
        """Update PPO : K epochs de mini-batch clipped surrogate + early stopping KL."""
        n = obs_t.shape[0]
        policy_losses, value_losses, kls = [], [], []

        for epoch in range(self.ppo_epochs):
            indices = torch.randperm(n)
            epoch_kls = []

            for start in range(0, n, self.minibatch_size):
                idx = indices[start:start + self.minibatch_size]
                obs_mb = obs_t[idx]
                act_mb = actions_t[idx]
                adv_mb = advantages_t[idx]
                ret_mb = returns_t[idx]
                old_lp_mb = old_log_probs_t[idx]

                log_probs_new, entropy = self.actor.evaluate(obs_mb, act_mb)
                ratio = torch.exp(log_probs_new - old_lp_mb)

                surr1 = ratio * adv_mb
                surr2 = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps) * adv_mb
                policy_loss = -torch.min(surr1, surr2).mean()
                entropy_loss = -entropy.mean()

                values_pred = self.critic(obs_mb)
                value_loss = F.mse_loss(values_pred, ret_mb)

                total_loss = (policy_loss
                              + self.value_coef * value_loss
                              + self.entropy_coef * entropy_loss)

                self.actor_optimizer.zero_grad()
                self.critic_optimizer.zero_grad()
                total_loss.backward()
                nn.utils.clip_grad_norm_(self.actor.parameters(), self.max_grad_norm)
                nn.utils.clip_grad_norm_(self.critic.parameters(), self.max_grad_norm)
                self.actor_optimizer.step()
                self.critic_optimizer.step()

                policy_losses.append(policy_loss.item())
                value_losses.append(value_loss.item())
                with torch.no_grad():
                    approx_kl = ((ratio - 1) - torch.log(ratio)).mean().item()
                    epoch_kls.append(approx_kl)

            kls.append(np.mean(epoch_kls))
            if np.mean(epoch_kls) > self.target_kl:
                break

        return {
            "policy_loss": np.mean(policy_losses),
            "value_loss": np.mean(value_losses),
            "kl": np.mean(kls),
        }

In [ ]:
# === Mini-test PPOAgent ===

torch.manual_seed(42)
ppo_test = PPOAgent(obs_dim=17, action_dim=6, n_steps=256)

print(f"PPOAgent :")
print(f"  Actor  : {sum(p.numel() for p in ppo_test.actor.parameters()):,} parametres")
print(f"  Critic : {sum(p.numel() for p in ppo_test.critic.parameters()):,} parametres")
print(f"  clip_eps={ppo_test.clip_eps}, ppo_epochs={ppo_test.ppo_epochs}, target_kl={ppo_test.target_kl}")
print()
print("Comparaison _update() :")
print("  TRPO : gradient -> CG (10 iter) -> line search -> set_flat_params (1 step)")
print("  PPO  : K epochs x mini-batches -> clip loss -> Adam.step() (K steps)")

In [ ]:
# === Comparaison KL : Adam non controle vs PPO clippe ===

torch.manual_seed(42)
np.random.seed(42)

n_demo = 256
obs_demo = torch.randn(n_demo, 17)
act_demo = torch.randn(n_demo, 6) * 0.5
adv_demo = torch.randn(n_demo)
adv_demo = (adv_demo - adv_demo.mean()) / (adv_demo.std() + 1e-8)

actor_ref = GaussianActor(17, 6)
with torch.no_grad():
    old_lp_ref, _ = actor_ref.evaluate(obs_demo, act_demo)
    dist_ref = actor_ref(obs_demo)
    mu_ref = dist_ref.mean.clone()
    std_ref = dist_ref.stddev.clone()


def compute_kl_after_step(actor_init_state, lr, use_clip):
    actor_clone = GaussianActor(17, 6)
    actor_clone.load_state_dict(actor_init_state)
    opt = optim.Adam(actor_clone.parameters(), lr=lr)
    lp, _ = actor_clone.evaluate(obs_demo, act_demo)
    ratio = torch.exp(lp - old_lp_ref.detach())
    if use_clip:
        surr = -torch.min(ratio * adv_demo,
                          torch.clamp(ratio, 0.8, 1.2) * adv_demo).mean()
    else:
        surr = -(ratio * adv_demo).mean()
    opt.zero_grad()
    surr.backward()
    opt.step()
    with torch.no_grad():
        d_new = actor_clone(obs_demo)
        kl = torch.distributions.kl_divergence(
            Normal(mu_ref, std_ref), d_new
        ).sum(-1).mean().item()
    return kl


init_state = actor_ref.state_dict()
lrs = [1e-4, 3e-4, 1e-3]
kl_adam_list = [compute_kl_after_step(init_state, lr, False) for lr in lrs]
kl_ppo_list  = [compute_kl_after_step(init_state, lr, True)  for lr in lrs]

x = np.arange(len(lrs))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width/2, kl_adam_list, width, label="Adam (pas de clip)", color="coral", alpha=0.85)
ax.bar(x + width/2, kl_ppo_list,  width, label="PPO (clip eps=0.2)", color="steelblue", alpha=0.85)
ax.axhline(0.01, color="black", linestyle="--", linewidth=2, label="delta = 0.01")
ax.set_xticks(x)
ax.set_xticklabels([f"lr={lr}" for lr in lrs])
ax.set_ylabel("KL apres 1 gradient step")
ax.set_title("Adam vs PPO : KL apres un gradient step (n=256)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("Valeurs KL :")
for lr, ka, kp in zip(lrs, kl_adam_list, kl_ppo_list):
    print(f"  lr={lr} : Adam={ka:.5f}  PPO={kp:.5f}  (seuil 0.01)")

## Interpretation : Adam viole la contrainte, PPO la respecte

Le graphique montre que pour un learning rate standard (3e-4) :

- **Adam sans clip** produit une KL bien superieure a $\delta = 0.01$ — meme avec un seul gradient step, la politique peut changer significativement
- **PPO clippe** maintient la KL proche ou sous le seuil — le clip agit comme un garde-fou naturel

Pour un learning rate eleve (1e-3), Adam produit une KL encore plus grande, tandis que PPO reste raisonnable grace au clip.

Ce n'est pas une garantie formelle : avec K epochs et un dataset favorable, PPO peut aussi violer la contrainte. Mais **empiriquement**, le clip combined a l'early stopping maintient la KL dans une plage raisonnable sans CG ni line search.

## Pseudocode TRPO et PPO (Spinning Up style)

Maintenant que les composantes sont claires, voici les deux algorithmes complets cote a cote.

$$\boxed{\begin{aligned}
&\textbf{TRPO} \\
&\text{Pour chaque iteration } k : \\
&\quad \mathcal{D}_k \leftarrow \text{trajectoires collectees avec } \pi_{\theta_k} \\
&\quad \hat{A}_t \leftarrow \text{GAE}(V_{\phi_k}) \\
&\quad \hat{g}_k \leftarrow \nabla_\theta \mathbb{E}[r_t(\theta)\hat{A}_t]_{\theta_k} \\
&\quad x_k \leftarrow \text{ConjugateGradient}(\hat{F}_k x = \hat{g}_k) \\
&\quad \theta_{k+1} \leftarrow \text{LineSearch}(\theta_k, x_k, \delta_{KL}) \\
&\quad \phi_{k+1} \leftarrow \arg\min_\phi\ \mathbb{E}[(V_\phi(s_t)-\hat{R}_t)^2]
\end{aligned}}
\qquad
\boxed{\begin{aligned}
&\textbf{PPO-Clip} \\
&\text{Pour chaque iteration } k : \\
&\quad \mathcal{D}_k \leftarrow \text{trajectoires collectees avec } \pi_{\theta_k} \\
&\quad \hat{A}_t \leftarrow \text{GAE}(V_{\phi_k}) \\
&\quad \text{Pour } K \text{ epochs de mini-batches :} \\
&\quad\quad r_t(\theta) \leftarrow \pi_\theta(a_t|s_t) / \pi_{\theta_k}(a_t|s_t) \\
&\quad\quad \theta \leftarrow \text{Adam sur } L^{CLIP}(\theta) \\
&\quad\quad \text{stop si } \bar{D}_{KL}(\pi_{\theta_k}, \pi_\theta) > \delta_{target} \\
&\quad \phi_{k+1} \leftarrow \arg\min_\phi\ \mathbb{E}[(V_\phi(s_t)-\hat{R}_t)^2]
\end{aligned}}$$

Les trois premieres lignes sont identiques : collecte, retours, avantages. La divergence est uniquement dans l'update actor : CG + line search (TRPO) vs Adam + clip + early stop (PPO).

## Note : normalisation des observations

PPO et TRPO controlent la **taille** des mises a jour de la politique. Mais aucun ne corrige automatiquement une mauvaise echelle des observations en entree du reseau.

Sur HalfCheetah, les 17 dimensions d'observation melangent positions articulaires ($\sim [-\pi, \pi]$), vitesses angulaires ($\pm 10$ rad/s) et coordonnees cartesiennes. Sans normalisation, les gradients sont mal conditionnes — meme si PPO clippe proprement le ratio.

En pratique, on applique une **normalisation en ligne** des observations :

$$\hat{s}_i = \text{clip}\!\left(\frac{s_i - \mu_i}{\sqrt{\sigma_i^2 + \epsilon}},\ -c,\ c\right)$$

Les statistiques $\mu_i, \sigma_i^2$ sont estimees pendant l'entrainement puis gelees pendant l'evaluation. Dans notre implementation simplifiee, on omet cette etape pour ne pas melanger les concepts. La mini-illustration suivante montre seulement l'effet de changement d'echelle sur un petit lot synthetique, sans lancer de training.

In [ ]:
# === Micro-illustration : effet d'une normalisation par dimension ===
rng = np.random.default_rng(7)
obs_demo = np.column_stack([
    rng.normal(0.0, 1.0, size=128),
    rng.normal(25.0, 12.0, size=128),
    rng.normal(-3.0, 0.15, size=128),
]).astype(np.float32)

mu = obs_demo.mean(axis=0)
std = obs_demo.std(axis=0) + 1e-8
obs_norm = np.clip((obs_demo - mu) / std, -5.0, 5.0)

print("Moyennes avant normalisation :", np.round(mu, 3))
print("Std avant normalisation      :", np.round(obs_demo.std(axis=0), 3))
print("Moyennes apres normalisation :", np.round(obs_norm.mean(axis=0), 3))
print("Std apres normalisation      :", np.round(obs_norm.std(axis=0), 3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for dim in range(obs_demo.shape[1]):
    axes[0].plot(obs_demo[:40, dim], label=f"dim {dim}")
    axes[1].plot(obs_norm[:40, dim], label=f"dim {dim}")
axes[0].set_title("Observations brutes")
axes[1].set_title("Observations normalisees")
for ax in axes:
    ax.legend()
    ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

**Lecture.** Avant normalisation, la deuxième dimension domine visuellement parce que son échelle est près de cent fois plus grande que celle de la troisième. Après centrage-réduction, les trois signaux ont une moyenne proche de zéro et un écart-type proche de un : l'acteur et le critique peuvent consacrer leur capacité aux relations entre variables plutôt qu'à compenser leurs unités. En pratique, ces statistiques doivent être estimées sur l'expérience d'entraînement puis figées pendant l'évaluation.

## Hyperparametres

On fixe les memes hyperparametres communs pour les deux algorithmes afin d'isoler l'effet de la methode d'update.

### Hyperparametres communs

| Parametre | Valeur | Role |
|-----------|--------|------|
| `hidden_dim` | 256 | Taille des couches cachees (standard MuJoCo) |
| `gamma` | 0.99 | Facteur d'actualisation |
| `n_steps` | 2048 | Longueur du rollout |
| `gae_lambda` | 0.95 | Parametre GAE |
| `lr_critic` | 3e-4 | Adam pour le critic |
| `max_grad_norm` | 0.5 | Gradient clipping (critic) |
| `total_timesteps` | 200 000 | Duree de l'entrainement |
| `seed` | 42 | Graine (identique pour les deux) |

### Hyperparametres specifiques a TRPO

| Parametre | Valeur | Role |
|-----------|--------|------|
| `max_kl` | 0.01 | Seuil de la trust region |
| `cg_steps` | 10 | Iterations du gradient conjugue |
| `cg_damping` | 0.1 | Regularisation Fisher-vecteur |
| `backtrack_coeff` | 0.5 | Facteur de reduction de la line search |
| `max_backtracks` | 10 | Max essais de la line search |
| `critic_epochs` | 5 | Epochs Adam pour le critic seul |

### Hyperparametres specifiques a PPO

| Parametre | Valeur | Role |
|-----------|--------|------|
| `lr_actor` | 3e-4 | Adam pour l'actor |
| `clip_eps` | 0.2 | Seuil de clipping du ratio |
| `ppo_epochs` | 10 | Epochs de mini-batch SGD par rollout |
| `minibatch_size` | 64 | Taille des mini-batches |
| `target_kl` | 0.01 | Seuil d'early stopping |

In [ ]:
# === Boucle d'entrainement commune ===

def train_agent(agent, env, total_timesteps, log_interval, label):
    """Boucle d'entrainement generique : collecte + update par rollout."""
    print(f"{'=' * 60}")
    print(f"ENTRAINEMENT {label}")
    print(f"{'=' * 60}")
    timesteps = 0
    last_log_t = 0

    while timesteps < total_timesteps:
        metrics = agent.learn_step(env)
        timesteps += agent.n_steps

        if timesteps - last_log_t >= log_interval or timesteps >= total_timesteps:
            last_log_t = timesteps
            print(
                f"Step {timesteps:>7d} | "
                f"Reward={metrics['mean_reward']:>8.1f} | "
                f"pi_loss={metrics['policy_loss']:>8.4f} | "
                f"v_loss={metrics['value_loss']:>8.4f} | "
                f"kl={metrics['kl']:>7.5f}"
            )

    print(f"\n{label} termine ({total_timesteps} steps).")

In [ ]:
# Configuration commune : TRPO et PPO voient exactement le même budget et les mêmes données.
SEED = 42
ENV_ID = "HalfCheetah-v5"
HIDDEN_DIM = 256
GAMMA = 0.99
N_STEPS = 512
GAE_LAMBDA = 0.95
LR_CRITIC = 3e-4
MAX_GRAD_NORM = 0.5
TOTAL_TIMESTEPS = 200_000
LOG_INTERVAL = 10_000

# Paramètres propres à TRPO.
MAX_KL = 0.01
CG_STEPS = 10
CG_DAMPING = 0.1
CRITIC_EPOCHS = 5

# Paramètres propres à PPO.
LR_ACTOR = 3e-4
CLIP_EPS = 0.2
PPO_EPOCHS = 10
MINIBATCH_SIZE = 64
TARGET_KL = 0.01

torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
# === Entrainement TRPO ===

torch.manual_seed(SEED)
np.random.seed(SEED)

env_trpo = gym.make(ENV_ID)
env_trpo.reset(seed=SEED)
obs_dim = env_trpo.observation_space.shape[0]
action_dim = env_trpo.action_space.shape[0]

agent_trpo = TRPOAgent(
    obs_dim=obs_dim, action_dim=action_dim,
    hidden_dim=HIDDEN_DIM, lr_critic=LR_CRITIC,
    gamma=GAMMA, n_steps=N_STEPS, gae_lambda=GAE_LAMBDA,
    max_grad_norm=MAX_GRAD_NORM,
    max_kl=MAX_KL, cg_steps=CG_STEPS, cg_damping=CG_DAMPING,
    critic_epochs=CRITIC_EPOCHS,
)

train_agent(agent_trpo, env_trpo, TOTAL_TIMESTEPS, LOG_INTERVAL, "TRPO")
env_trpo.close()

In [ ]:
# === Entrainement PPO ===

torch.manual_seed(SEED)
np.random.seed(SEED)

env_ppo = gym.make(ENV_ID)
env_ppo.reset(seed=SEED)

agent_ppo = PPOAgent(
    obs_dim=obs_dim, action_dim=action_dim,
    hidden_dim=HIDDEN_DIM, lr_critic=LR_CRITIC,
    gamma=GAMMA, n_steps=N_STEPS, gae_lambda=GAE_LAMBDA,
    max_grad_norm=MAX_GRAD_NORM,
    lr_actor=LR_ACTOR, clip_eps=CLIP_EPS,
    ppo_epochs=PPO_EPOCHS, minibatch_size=MINIBATCH_SIZE,
    target_kl=TARGET_KL,
)

train_agent(agent_ppo, env_ppo, TOTAL_TIMESTEPS, LOG_INTERVAL, "PPO")
env_ppo.close()

In [ ]:
# === Courbes d'apprentissage comparatives ===

def moving_average(data, window=5):
    if len(data) < window:
        return np.array(data)
    return np.convolve(data, np.ones(window) / window, mode="valid")


fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("TRPO vs PPO sur HalfCheetah-v5 (200k steps)", fontsize=14, fontweight="bold")

n_iters_trpo = len(agent_trpo.history["rewards"])
n_iters_ppo = len(agent_ppo.history["rewards"])
x_trpo = np.arange(n_iters_trpo) * N_STEPS / 1000
x_ppo = np.arange(n_iters_ppo) * N_STEPS / 1000
window = 3
c_trpo, c_ppo = "steelblue", "coral"

keys = ["rewards", "policy_loss", "kl", "value_loss"]
titles = ["Recompense moyenne", "Policy Loss", "KL divergence", "Value Loss (MSE)"]

for ax, key, title in zip(axes.flat, keys, titles):
    d_trpo = agent_trpo.history[key]
    d_ppo = agent_ppo.history[key]

    ax.plot(x_trpo, d_trpo, alpha=0.2, color=c_trpo)
    ax.plot(x_ppo, d_ppo, alpha=0.2, color=c_ppo)

    ma_trpo = moving_average(d_trpo, window)
    ma_ppo = moving_average(d_ppo, window)
    x_ma_t = x_trpo[window - 1:window - 1 + len(ma_trpo)]
    x_ma_p = x_ppo[window - 1:window - 1 + len(ma_ppo)]

    ax.plot(x_ma_t, ma_trpo, color=c_trpo, linewidth=2, label="TRPO")
    ax.plot(x_ma_p, ma_ppo, color=c_ppo, linewidth=2, label="PPO")

    if key == "kl":
        ax.axhline(MAX_KL, color="red", linestyle="--", alpha=0.5,
                   linewidth=1, label=f"delta={MAX_KL}")

    ax.set_xlabel("Timesteps (x1000)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Interpretation : lecture des courbes

### Panel 1 : Recompense moyenne (axe x = timesteps, axe y = recompense par episode)

Dans ce protocole, PPO reutilise chaque rollout pendant `PPO_EPOCHS=10`, alors que TRPO fait une seule mise a jour actor par rollout puis passe a la collecte suivante. Cela donne a PPO davantage d'occasions d'ajuster actor et critic pour un meme budget de transitions. Sur nos courbes, cela se traduit souvent par une montee plus rapide et plus reguliere, sans que ce soit une loi generale.

TRPO peut parfois atteindre de meilleures recompenses ponctuelles, mais avec plus de variance d'une seed a l'autre et d'un bloc d'updates au suivant.

### Panel 2 : Policy Loss (axe y = loss surrogate)

La policy loss ne mesure pas directement la qualite de la politique. Elle peut augmenter quand l'agent s'ameliore : les cibles deviennent plus ambitieuses. Ce qui compte : la loss ne doit pas diverger. PPO a souvent une loss plus nerveuse (plusieurs epochs, mini-batches differents), mais le clipping doit eviter les emballements durables.

### Panel 3 : KL divergence (axe y = KL, ligne rouge = seuil delta=0.01)

TRPO est concu pour rester proche du seuil `max_kl=0.01`, mais la contrainte reste numerique et approximee : on cherche surtout a voir une KL contenue, pas une egalite parfaite a chaque update. PPO peut occasionnellement depasser `target_kl`, puis l'early stopping limite le nombre d'epochs supplementaires.

### Panel 4 : Value Loss (axe y = MSE du critique)

Les deux algorithmes entrainent le critic avec Adam. PPO retravaille plus longtemps le meme rollout, ce qui peut aider le critic a suivre plus vite; TRPO garde en contrepartie une separation plus nette entre update actor et update critic.

In [ ]:
# === Tableau de metriques finales ===

print("Metriques finales (moyenne sur les 10 dernieres iterations) :")
print()
print(f"{'Metrique':>25} | {'TRPO':>12} | {'PPO':>12} | {'Gagnant':>10}")
print("-" * 70)

metrics_compare = {
    "Recompense moyenne": ("rewards", "max"),
    "Policy Loss": ("policy_loss", "none"),
    "KL divergence": ("kl", "none"),
    "Value Loss": ("value_loss", "min"),
}

for label, (key, winner_dir) in metrics_compare.items():
    v_t = np.mean(agent_trpo.history[key][-10:])
    v_p = np.mean(agent_ppo.history[key][-10:])
    if winner_dir == "max":
        winner = "PPO" if v_p > v_t else "TRPO"
    elif winner_dir == "min":
        winner = "PPO" if v_p < v_t else "TRPO"
    else:
        winner = "---"
    print(f"{label:>25} | {v_t:>12.4f} | {v_p:>12.4f} | {winner:>10}")

print()
n_updates_trpo = len(agent_trpo.history["rewards"])
n_updates_ppo = len(agent_ppo.history["rewards"]) * PPO_EPOCHS
print(f"Gradient steps actor :")
print(f"  TRPO : {n_updates_trpo} (1 step CG+LS par rollout)")
print(f"  PPO  : ~{n_updates_ppo} ({PPO_EPOCHS} epochs x {n_updates_ppo // PPO_EPOCHS} rollouts)")
print(f"  PPO fait environ {n_updates_ppo // n_updates_trpo}x plus d'updates actor")

## Recapitulatif TRPO vs PPO

| Dimension | TRPO | PPO |
|-----------|------|-----|
| **Contrainte KL** | Visee explicitement par line search | Implicite via clipping + early stop |
| **Optimiseur actor** | Gradient conjugue + backtracking | Adam standard |
| **Epochs par rollout** | 1 | K (typiquement 10) |
| **Cout computationnel** | Plus eleve a iso-architecture | Plus simple en pratique |
| **Efficacite en donnees** | Limitee par une seule mise a jour actor | Meilleure reutilisation d'un meme rollout |
| **Garantie theorique** | Plus forte sur le papier | Plus faible mais robuste en pratique |
| **Implementation** | Complexe (CG, Fisher, LS) | Simple (loss clippee + Adam) |
| **Runs archives de ce chapitre (1M, 3 seeds)** | moyenne plus haute mais tres dispersee | moyenne plus basse mais plus resserree |
| **Lecture prudente** | sensible aux seeds et aux details numeriques | compromis pratique souvent prefere |

Les chiffres archives pour ce chapitre suggerent que TRPO peut obtenir de tres bons pics, mais avec une variance enorme. PPO converge de facon plus previsible dans cette implementation. Cette lecture vaut pour ces runs et cette recette, pas comme classement universel de tous les setups TRPO/PPO.

## TRPO/PPO vs A2C : pourquoi PPO est souvent le compromis pratique

La progression des algorithmes dans ce notebook :

$$\text{A2C} \xrightarrow{+\text{trust region exacte}} \text{TRPO} \xrightarrow{+\text{simplification}} \text{PPO}$$

| Algorithme | Mecanisme de stabilite | Prix |
|-----------|------------------------|------|
| A2C | Gradient clipping (norme) | Aucun |
| TRPO | Contrainte KL explicite | CG + line search |
| PPO | Clipping du ratio + early stop | Quelques epochs supplementaires |

**Pourquoi pas A2C ?** A2C n'a aucune contrainte directe sur le changement de politique. Sur HalfCheetah, un mauvais rollout peut suffire a faire un grand pas et a degrader fortement la politique.

**Pourquoi pas TRPO ?** Cout computationnel plus eleve et implementation plus delicate (CG, damping, line search). Cela paie parfois, mais le benefice depend fortement du setup.

**Pourquoi PPO ?** Dans cette famille d'algorithmes, PPO capture une grande partie de l'idee de trust region avec une implementation plus legere. C'est pour cela qu'il a longtemps servi de choix par defaut dans beaucoup de bibliotheques et de projets appliques.

| Algorithme | Best eval | Final eval | Variance inter-seed |
|------------|-----------|------------|---------------------|
| A2C | 326 +/- 350 | -181 +/- 257 | Tres forte |
| A2C-GAE | 2317 +/- 981 | 2092 +/- 1253 | Forte |
| PPO | 1568 +/- 33 | 1548 +/- 35 | Tres faible |
| TRPO | 3055 +/- 2080 | 2858 +/- 1814 | Tres forte |

Ces chiffres sont des ordres de grandeur issus des runs de ce projet. Ils servent ici a montrer les profils de stabilite observes, pas a etablir un leaderboard definitif entre algorithmes.

In [ ]:
# === Démonstration finale : replay vidéo des politiques déterministes ===
def evaluate_and_display_agent(agent, label, n_episodes=5, seed=0, video_root="videos/06_trpo_ppo_halfcheetah"):
    """Évalue la politique déterministe et affiche le dernier replay vidéo enregistré."""
    safe_label = label.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    env = gym.make(ENV_ID, render_mode="rgb_array")
    env = RecordVideo(
        env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == n_episodes - 1,
        name_prefix=f"{safe_label}_greedy",
    )

    agent.actor.eval()
    demo_rewards = []

    print(f"{'=' * 50}")
    print(f"Démo greedy : {label}")
    print(f"{'=' * 50}")

    try:
        for ep in range(n_episodes):
            obs, _ = env.reset(seed=seed + ep)
            total = 0.0
            steps = 0
            done = False

            while not done:
                with torch.no_grad():
                    obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                    dist = agent.actor(obs_t)
                    action = dist.mean.squeeze(0).numpy()
                    action = np.clip(action, env.action_space.low, env.action_space.high)

                obs, reward, terminated, truncated, _ = env.step(action)
                total += reward
                steps += 1
                done = terminated or truncated

            demo_rewards.append(total)
            print(f"[{label}] Épisode {ep + 1} : reward={total:.1f} ({steps} pas)")

    finally:
        env.close()

    print(f"[{label}] Moyenne : {np.mean(demo_rewards):.1f} +/- {np.std(demo_rewards):.1f}")

    videos = sorted(video_dir.glob("*.mp4"), key=lambda path: path.stat().st_mtime)
    if videos and Video is not None and display is not None:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=520))
    elif not videos:
        print(f"Aucune vidéo générée pour {label}.")
    else:
        print(f"Vidéo générée mais affichage IPython indisponible : {videos[-1]}")

    return demo_rewards


demo_trpo = evaluate_and_display_agent(agent_trpo, "TRPO")
demo_ppo = evaluate_and_display_agent(agent_ppo, "PPO")

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(demo_trpo))
width = 0.35
ax.bar(x - width / 2, demo_trpo, width, label="TRPO", color="steelblue", alpha=0.85)
ax.bar(x + width / 2, demo_ppo, width, label="PPO", color="coral", alpha=0.85)
ax.axhline(np.mean(demo_trpo), color="steelblue", linestyle="--", linewidth=1.5, label=f"TRPO moy = {np.mean(demo_trpo):.1f}")
ax.axhline(np.mean(demo_ppo), color="coral", linestyle="--", linewidth=1.5, label=f"PPO moy = {np.mean(demo_ppo):.1f}")
ax.set_xlabel("Épisode")
ax.set_ylabel("Récompense totale")
ax.set_title("Démo greedy : 5 épisodes sans exploration (action = mu de la gaussienne)")
ax.set_xticks(x)
ax.set_xticklabels([f"Ep {i + 1}" for i in x])
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


## Interpretation : demo greedy

En mode greedy, l'agent prend l'action $a = \mu_\theta(s)$ (la moyenne de la gaussienne, sans echantillonnage). C'est la politique deterministe optimale apprise.

**Axe x** : episodes de test (1 a 5), chacun independant (seed differente)
**Axe y** : recompense totale sur 1000 pas — correspond directement a la vitesse moyenne de course moins le cout d'effort

**Ce qu'on cherche** : des barres hautes et regulieres. Une grande variance inter-episodes indique une politique fragile qui fonctionne bien sur certains etats initiaux mais pas d'autres.

Apres seulement 200k steps, les politiques apprises sont loin du score optimal (~3000-6000 pour 1M steps), mais elles montrent deja la difference entre un robot qui commence a courir (recompense positive) et un comportement quasi-aleatoire (recompense proche de zero).

## Conclusion

### Ce qu'on a construit

Deux algorithmes **trust region** sur HalfCheetah, construits a partir d'une base commune :

- **TRPO** : resout exactement le probleme contraint $\max L$ s.t. $D_{KL} \leq \delta$ via gradient conjugue et backtracking line search. Garantie theorique de monotonie. Cout computationnel eleve.
- **PPO** : approxime la contrainte par clipping du ratio $r_t(\theta) \in [1-\varepsilon, 1+\varepsilon]$. K epochs de mini-batch Adam. Simple, efficace, standard industriel.

Les briques fondamentales introduites :

| Brique | Role |
|--------|------|
| Ratio $r_t(\theta)$ | Pont entre importance sampling et policy gradient |
| Produit Fisher-vecteur | Calculer $Fv$ en $O(|\theta|)$ sans former $F$ |
| Gradient conjugue | Resoudre $Fx = g$ de facon iterative |
| Backtracking line search | Step adaptatif avec verification de contrainte |
| Loss clippee PPO | $\min(r\hat{A}, \text{clip}(r)\hat{A})$ |

### Vers les methodes off-policy : SAC et TD3

Tous les algorithmes vus jusqu'ici (A2C, TRPO, PPO) sont **on-policy** : les donnees utilisees pour l'update viennent toujours de la politique courante. Quand l'interaction avec l'environnement est couteuse, c'est inefficace — chaque transition n'est utilisee qu'une seule fois.

Les methodes **off-policy** (DDPG, TD3, SAC) stockent toutes les transitions dans un replay buffer et peuvent apprendre de donnees collectees par n'importe quelle politique. SAC ajoute la notion d'entropie maximale : la politique apprend a etre la plus performante possible tout en restant la plus aleatoire possible, ce qui ameliore l'exploration et la robustesse.

Le prochain notebook introduit SAC sur HalfCheetah, avec la politique tanh-squashee qui borne naturellement les actions.

## References

- Schulman, J., Levine, S., Abbeel, P., Jordan, M. & Moritz, P. (2015). Trust Region Policy Optimization. [arXiv:1502.05477](https://arxiv.org/abs/1502.05477).
- Schulman, J., Wolski, F., Dhariwal, P., Radford, A. & Klimov, O. (2017). Proximal Policy Optimization Algorithms. [arXiv:1707.06347](https://arxiv.org/abs/1707.06347).
- Schulman, J., Moritz, P., Levine, S., Jordan, M. & Abbeel, P. (2016). High-Dimensional Continuous Control Using Generalized Advantage Estimation. [arXiv:1506.02438](https://arxiv.org/abs/1506.02438).
- Haarnoja, T. et al. (2018). Soft Actor-Critic. [arXiv:1801.01290](https://arxiv.org/abs/1801.01290).
- Spinning Up in Deep RL. OpenAI. [spinningup.openai.com](https://spinningup.openai.com/en/latest/).